# 🎼 Python Orchestration — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> A DAG is a recipe. Each task is one cooking step. Airflow is the kitchen manager — they read the recipe, figure out which steps can happen in parallel, assign workers, and track when each step finishes. If a step fails (the sauce burns), the manager retries it, and alerts the chef. An idempotent task is one where you can repeat it as many times as you want and always get the same dish — even if the kitchen lost power mid-step.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is Orchestration? The Visual Model](#1) |
| 2 | [Core Concepts — Setup](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: DAG Design — Tasks, Operators, Dependencies](#5) |
| 6 | [Pattern 2: Idempotency — Safe Retries](#6) |
| 7 | [Pattern 3: Retry & SLA Alerting](#7) |
| 8 | [Pattern 4: Dynamic DAGs — Factory Pattern](#8) |
| 9 | [Pattern 5: Sensor Patterns & External Triggers](#9) |
| 10 | [The Orchestration Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>
## 1. What Is Orchestration? The Visual Model

---

```
AIRFLOW DAG: nightly_etl

  [check_source_file]   [validate_schema]      ← sensors / validators
          │                    │
          └─────────┬──────────┘
                    │
              [extract]                        ← data extraction
                    │
     ┌──────────────┼──────────────┐
     │              │              │
 [transform_A]  [transform_B]  [transform_C]  ← parallel transforms
     │              │              │
     └──────────────┼──────────────┘
                    │
                [load]                         ← write to warehouse
                    │
              [notify_slack]                   ← downstream trigger

DAG CONCEPTS:
  DAG:        Directed Acyclic Graph — no cycles, tasks flow one direction
  Task:       one unit of work (Python function, SQL query, Spark job)
  Operator:   task template (PythonOperator, BashOperator, SparkSubmitOperator)
  Executor:   runs tasks (LocalExecutor, CeleryExecutor, KubernetesExecutor)
  Task State: queued → running → success / failed / retry / skipped

TASK INSTANCE STATE MACHINE:
  scheduled → queued → running → { success | failed }
                                       │
                                    retry (if retries > 0)
                                       │
                                    failed (if max retries exceeded)

CRITICAL DATE CONCEPTS:
  execution_date:  the DATA date this run processes (not when it runs)
  data_interval:   [execution_date, execution_date + schedule_interval)
  start_date:      when Airflow starts scheduling this DAG
  catchup=True:    backfill all missed runs since start_date
```


<a id='2'></a>
## 2. Core Concepts — Setup

In [ ]:
import time
import random
import hashlib
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Callable, Any
from enum import Enum
from collections import defaultdict, deque

random.seed(42)

class TaskState(Enum):
    PENDING  = 'pending'
    QUEUED   = 'queued'
    RUNNING  = 'running'
    SUCCESS  = 'success'
    FAILED   = 'failed'
    RETRY    = 'retry'
    SKIPPED  = 'skipped'
    UPSTREAM_FAILED = 'upstream_failed'

@dataclass
class TaskDef:
    task_id:       str
    operator:      str  # 'python', 'bash', 'spark', 'sql', 'sensor'
    retries:       int  = 3
    retry_delay_s: int  = 300  # 5 minutes
    sla_minutes:   Optional[int] = None
    depends_on:    List[str] = field(default_factory=list)

@dataclass
class TaskInstance:
    task_id:      str
    execution_date: str
    state:        TaskState = TaskState.PENDING
    attempts:     int  = 0
    start_time:   Optional[float] = None
    end_time:     Optional[float] = None
    xcom_value:   Any = None  # cross-task communication value

class SimpleDAG:
    def __init__(self, dag_id, schedule_interval='@daily', catchup=False):
        self.dag_id = dag_id
        self.schedule_interval = schedule_interval
        self.catchup = catchup
        self.tasks: Dict[str, TaskDef] = {}

    def add_task(self, task: TaskDef):
        self.tasks[task.task_id] = task

    def topological_order(self) -> List[str]:
        # Kahn's algorithm — find valid execution order
        in_degree = {tid: 0 for tid in self.tasks}
        for t in self.tasks.values():
            for dep in t.depends_on:
                in_degree[t.task_id] += 1
        queue = deque([tid for tid, d in in_degree.items() if d == 0])
        order = []
        while queue:
            tid = queue.popleft()
            order.append(tid)
            for t in self.tasks.values():
                if tid in t.depends_on:
                    in_degree[t.task_id] -= 1
                    if in_degree[t.task_id] == 0:
                        queue.append(t.task_id)
        return order

# build example DAG
dag = SimpleDAG('nightly_etl', '@daily')
for t in [
    TaskDef('check_file',    'sensor', retries=5, sla_minutes=30),
    TaskDef('extract',       'python', retries=3, depends_on=['check_file']),
    TaskDef('transform_A',   'spark',  retries=2, depends_on=['extract']),
    TaskDef('transform_B',   'spark',  retries=2, depends_on=['extract']),
    TaskDef('load',          'sql',    retries=3, depends_on=['transform_A', 'transform_B']),
    TaskDef('notify',        'python', retries=1, depends_on=['load']),
]:
    dag.add_task(t)

print(f"DAG '{dag.dag_id}': {len(dag.tasks)} tasks")
print(f"Execution order: {dag.topological_order()}")
print("Setup complete.")

<a id='3'></a>
## 3. The Core API — All Operations

---

```
AIRFLOW OPERATIONS
─────────────────────────────────────────────────────────────────────────
OPERATION                       WHAT IT DOES
─────────────────────────────────────────────────────────────────────────
DAG(dag_id, schedule_interval)  define DAG: schedule, start_date, catchup
PythonOperator(python_callable) run a Python function as a task
BashOperator(bash_command)      run a shell command
SparkSubmitOperator(...)        submit a Spark job to YARN/K8s
BigQueryOperator(sql)           run SQL in BigQuery
task_A >> task_B                set B as downstream of A (dependency)
[t_A, t_B] >> t_C              C depends on both A and B
Variable.get(key)               fetch Airflow variable (secrets/config)
XCom.push(key, value)           push value for downstream tasks to read
XCom.pull(task_id, key)         read value pushed by upstream task
trigger_rule='all_done'         run even if upstream failed
trigger_rule='one_success'      run if any upstream succeeds
on_failure_callback=fn          call fn when task fails
sla=timedelta(minutes=30)       alert if task exceeds 30 min
─────────────────────────────────────────────────────────────────────────

THINGS YOU DO NOT DO:
❌  Use execution_date as wall-clock time — it's the DATA date, not run time
❌  Write side effects before the idempotency check — always check first
❌  Use catchup=True without understanding backfill implications
❌  Pass large objects via XCom — XCom is for small metadata, not DataFrames
❌  Set retries=0 on network-dependent tasks — transient failures are real
❌  Hardcode connection strings in DAG code — use Airflow Connections
```


In [ ]:
# Core API demo: simulate DAG execution with state machine

class DAGRunner:
    def __init__(self, dag: SimpleDAG):
        self.dag = dag
        self.instances: Dict[str, TaskInstance] = {}
        self.xcom_store: Dict[str, Any] = {}

    def run(self, execution_date: str, fail_tasks: List[str] = []):
        order = self.dag.topological_order()
        for tid in order:
            task_def = self.dag.tasks[tid]
            # check if any upstream failed
            upstream_ok = all(
                self.instances.get(dep, TaskInstance(dep, execution_date)).state == TaskState.SUCCESS
                for dep in task_def.depends_on
            )
            ti = TaskInstance(tid, execution_date)
            if not upstream_ok:
                ti.state = TaskState.UPSTREAM_FAILED
                self.instances[tid] = ti
                print(f"  {tid:20s}: {ti.state.value} (upstream failed)")
                continue
            # simulate task execution
            ti.state = TaskState.RUNNING
            ti.start_time = time.time()
            if tid in fail_tasks:
                ti.state = TaskState.FAILED
                ti.attempts = task_def.retries + 1
            else:
                ti.state = TaskState.SUCCESS
                ti.attempts = 1
                ti.xcom_value = f"{tid}_output_{execution_date}"
                self.xcom_store[tid] = ti.xcom_value
            ti.end_time = time.time()
            self.instances[tid] = ti
            attempt_str = f"(attempt {ti.attempts})"
            print(f"  {tid:20s}: {ti.state.value:20s} {attempt_str}")

    def summary(self):
        counts = defaultdict(int)
        for ti in self.instances.values():
            counts[ti.state.value] += 1
        print(f"  Summary: {dict(counts)}")

print("=== Normal run (all succeed) ===")
runner1 = DAGRunner(dag)
runner1.run('2024-03-15')
runner1.summary()

print()
print("=== Failed run (transform_A fails) ===")
runner2 = DAGRunner(dag)
runner2.run('2024-03-15', fail_tasks=['transform_A'])
runner2.summary()
print("  (transform_A fails → load + notify skipped via upstream_failed)")

print("\nCore API demo complete.")

<a id='4'></a>
## 4. Decision Map — When To Use What

---

```
SIGNAL IN THE PROBLEM                        WHAT TO DO
────────────────────────────────────────────────────────────────────────────
Task can run in parallel                     split into independent tasks
Task must run after N others                 set upstream dependencies
Task needs upstream output                   XCom push/pull
Same logic for 20 tables                     dynamic DAG factory
Wait for external file to arrive             FileSensor / S3KeySensor
Wait for another DAG to finish               ExternalTaskSensor
Alert if ETL > 2 hours                       sla=timedelta(hours=2)
Re-run failed task without re-running all    airflow tasks clear + re-run
Task fails halfway — avoid re-processing     idempotent write (overwrite/upsert)
Secrets in DAG code                          Airflow Variable / Secrets Manager
────────────────────────────────────────────────────────────────────────────
```


<a id='5'></a>
## 5. 🧩 Pattern 1: DAG Design — Tasks, Operators, Dependencies

---

```
PROBLEM:
  Design a DAG that: validates input files, runs 3 parallel transforms,
  loads results, then sends a Slack notification.

APPROACH:
  Decompose work into atomic tasks. Set dependencies to maximize parallelism.
  Task granularity rule: one task = one logical step, one retry boundary.

TASK DECOMPOSITION PRINCIPLES:
  1. One task = one retry boundary (don't combine steps with different failure modes)
  2. Maximize parallelism — independent steps in parallel tasks
  3. Idempotent tasks — safe to re-run without side effects
  4. Fast tasks first — fail early, don't waste time if input is bad

DEPENDENCY PATTERNS:
  Sequential:   A >> B >> C
  Fan-out:      A >> [B, C, D]
  Fan-in:       [B, C, D] >> E
  Diamond:      A >> [B,C] >> D (join)
  Cross:        [A,B] >> [C,D] (all combinations)

TRIGGER RULES:
  all_success (default):  run only if ALL upstream tasks succeed
  all_done:               run regardless of upstream outcome
  all_failed:             run only if ALL upstream failed (alert tasks)
  one_success:            run if any upstream succeeds (OR logic)
  one_failed:             run if any upstream fails (error handler)
  none_failed:            run if no upstream failed (skipped is ok)

KEY INSIGHT:
  Fan-out before fan-in = maximum parallelism.
  Use trigger_rule='all_done' for cleanup/notification tasks so they
  run even when upstream fails — ensures alerts always fire.

TIME / SPACE:
  DAG planning: O(T + E) — T tasks, E edges (topological sort)
  Parallel tasks limited by: concurrency config + available workers
```


In [ ]:
# Pattern 1: DAG design and execution

# Slow motion: build and execute the nightly ETL DAG
# step 1: define tasks with operators and dependencies
# step 2: topological sort → execution order
# step 3: execute in order, respecting parallelism where upstream is satisfied
# step 4: propagate failure state to downstream via upstream_failed

class ParallelDAGRunner:
    """
    Orchestration Pattern 1 — DAG execution with parallelism.
    Approach: Topological sort + level-by-level parallel execution.
    Time:  O(T + E) for scheduling, O(wall_time) = O(longest_path)
    Space: O(T) for task instances
    """
    def __init__(self, dag):
        self.dag = dag

    def get_levels(self):
        # compute which tasks can run in parallel (same dependency level)
        levels = {}
        for tid, task in self.dag.tasks.items():
            if not task.depends_on:
                levels[tid] = 0
            else:
                levels[tid] = max(levels.get(dep, 0) for dep in task.depends_on) + 1
        # group by level
        by_level = defaultdict(list)
        for tid, lvl in levels.items():
            by_level[lvl].append(tid)
        return dict(sorted(by_level.items()))

    def estimate_walltime(self, task_durations):
        # critical path = longest path through DAG
        levels = self.get_levels()
        cumulative = {}
        for lvl, tids in levels.items():
            level_max = max(task_durations.get(tid, 1) for tid in tids)
            prev_max  = max((cumulative.get(dep, 0)
                            for t in tids
                            for dep in self.dag.tasks[t].depends_on), default=0)
            for tid in tids:
                cumulative[tid] = prev_max + task_durations.get(tid, 1)
        return max(cumulative.values())

# build the nightly ETL DAG with realistic durations
etl_dag = SimpleDAG('nightly_etl_v2', '@daily')
tasks_defs = [
    TaskDef('validate_source', 'sensor', retries=5, sla_minutes=30),
    TaskDef('extract',         'python', retries=3, depends_on=['validate_source']),
    TaskDef('transform_sales', 'spark',  retries=2, depends_on=['extract']),
    TaskDef('transform_users', 'spark',  retries=2, depends_on=['extract']),
    TaskDef('transform_events','spark',  retries=2, depends_on=['extract']),
    TaskDef('load_warehouse',  'sql',    retries=3, depends_on=['transform_sales','transform_users','transform_events']),
    TaskDef('update_catalog',  'python', retries=2, depends_on=['load_warehouse']),
    TaskDef('notify_complete', 'python', retries=1, depends_on=['load_warehouse']),
]
for t in tasks_defs:
    etl_dag.add_task(t)

runner = ParallelDAGRunner(etl_dag)
levels = runner.get_levels()

print("=== DAG Execution Levels (parallel execution groups) ===")
for lvl, tids in levels.items():
    print(f"  Level {lvl}: {tids}  {'← parallel' if len(tids) > 1 else ''}")

# estimate wall time with and without parallelism
task_minutes = {
    'validate_source': 5,  'extract': 20,
    'transform_sales': 30, 'transform_users': 25, 'transform_events': 40,
    'load_warehouse': 15,  'update_catalog': 5,   'notify_complete': 1,
}
wall_time = runner.estimate_walltime(task_minutes)
sequential = sum(task_minutes.values())

print()
print("=== Wall Time: Parallel vs Sequential ===")
print(f"  Sequential (no parallelism): {sequential} min")
print(f"  Parallel (current DAG):      ~{wall_time} min")
print(f"  Speedup: {sequential/wall_time:.1f}×  (3 transforms run in parallel)")

print()
print("=== Trigger Rule Scenarios ===")
rules = [
    ('all_success (default)', 'notify', 'runs only if all upstream succeeded'),
    ('all_done',              'notify', 'runs even if transform failed — ensures alert fires'),
    ('one_failed',            'alert',  'dedicated error handler task'),
    ('none_failed',           'downstream', 'skips are OK, but failures block'),
]
for rule, task, description in rules:
    print(f"  trigger_rule='{rule:20s}' → {description}")

print("\nDAG design pattern complete.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Idempotency — Safe Retries

---

```
PROBLEM:
  A task writes 1M rows to a database, then crashes after row 600k.
  On retry, it writes again — now you have 600k duplicate rows.
  How do you design the task to be safe on retry?

APPROACH:
  Idempotent task: running it N times produces the same result as running once.
  Two strategies:
    1. OVERWRITE: write to a staging partition, then overwrite final target
    2. UPSERT/MERGE: use a natural key — re-running inserts or updates, never duplicates

OVERWRITE PATTERN:
  Step 1: write transformed data to staging: stage/date=2024-03-15/
  Step 2: atomic overwrite final: REPLACE INTO final/ WHERE date='2024-03-15'
  On retry: stage is overwritten (step 1) then final is overwritten (step 2)
  Result: identical to first successful run — idempotent

UPSERT PATTERN:
  SQL: INSERT INTO target ON CONFLICT(key) DO UPDATE SET ...
  Spark: df.write.format('delta').mode('overwrite').option('replaceWhere', 'date=...')
  Key requirement: natural dedup key must exist (event_id, transaction_id)

HIGH WATER MARK PATTERN:
  Store last_processed_id or last_processed_timestamp in a state table.
  On every run: SELECT MAX(id) FROM state → start from that point.
  On retry: re-reads from last committed checkpoint — no duplicates.

KEY INSIGHT:
  If your task can't be idempotent, make it atomic:
  write to temp → validate → rename/swap (atomic at filesystem level).
  Never partial-write to production. Write temp, swap atomically.

TIME / SPACE:
  Overwrite: O(N) — write N rows, delete old partition, write new
  Upsert:    O(N log N) — index lookup per row for conflict detection
  Both:      Safe to retry → reduced incident response time
```


In [ ]:
# Pattern 2: Idempotency simulation

# Slow motion: non-idempotent vs idempotent task execution with crash
# non-idempotent: append to table → retry appends again → duplicates
# idempotent (overwrite): write to partition → on retry, overwrites same partition
# idempotent (upsert): on conflict(key) update → retry updates same rows

class TargetTable:
    def __init__(self, name):
        self.name = name
        self.rows: Dict[str, dict] = {}  # key → row
        self.partitions: Dict[str, List[dict]] = defaultdict(list)

    def append(self, rows):
        for row in rows:
            self.rows[f"{self.name}_{len(self.rows)}"] = row  # no dedup

    def overwrite_partition(self, partition_key, rows):
        self.partitions[partition_key] = list(rows)  # atomic replace

    def upsert(self, rows, key_col):
        for row in rows:
            pk = row[key_col]  # natural key
            self.rows[pk] = row  # insert or update — idempotent

    def total_rows(self):
        return len(self.rows) + sum(len(v) for v in self.partitions.values())

def make_source_rows(date, n=1000):
    return [{'event_id': f'{date}_{i}', 'date': date, 'value': i * 10} for i in range(n)]

source_rows = make_source_rows('2024-03-15', n=100)

print("=== Non-Idempotent Task (append) ===")
bad_table = TargetTable('orders_bad')
print("  Run 1 (normal):")
bad_table.append(source_rows)
print(f"    rows after run 1: {bad_table.total_rows()}")

print("  Run 2 (retry after crash):")
bad_table.append(source_rows)  # crash recovery → appends again
print(f"    rows after retry: {bad_table.total_rows()}  ← DUPLICATES!")

print()
print("=== Idempotent Task (partition overwrite) ===")
good_table_ovr = TargetTable('orders_overwrite')
partition = '2024-03-15'
print("  Run 1 (normal):")
good_table_ovr.overwrite_partition(partition, source_rows)
print(f"    rows after run 1: {good_table_ovr.total_rows()}")

print("  Run 2 (retry after crash):")
good_table_ovr.overwrite_partition(partition, source_rows)  # replaces same partition
print(f"    rows after retry: {good_table_ovr.total_rows()}  ← SAME — idempotent")

print()
print("=== Idempotent Task (upsert on natural key) ===")
good_table_ups = TargetTable('orders_upsert')
print("  Run 1 (normal):")
good_table_ups.upsert(source_rows, 'event_id')
print(f"    rows after run 1: {good_table_ups.total_rows()}")

print("  Run 2 (retry — same data):")
good_table_ups.upsert(source_rows, 'event_id')  # ON CONFLICT → UPDATE (no new rows)
print(f"    rows after retry: {good_table_ups.total_rows()}  ← SAME — idempotent")

print()
print("=== High Water Mark Pattern ===")
state_table = {'last_processed_id': 0}  # persisted checkpoint

def idempotent_incremental(source_data, state):
    hwm = state['last_processed_id']
    new_rows = [r for r in source_data if r['event_id'] > hwm]  # only new rows
    if new_rows:
        max_id = max(r['event_id'] for r in new_rows)
        state['last_processed_id'] = max_id  # commit checkpoint
    return new_rows

int_source = [{'event_id': i, 'value': i*10} for i in range(1, 21)]
batch1 = idempotent_incremental(int_source, state_table)
print(f"  First run: processed {len(batch1)} rows, hwm={state_table['last_processed_id']}")
# simulate crash after partial run — next run restarts from hwm
batch2 = idempotent_incremental(int_source, state_table)  # already processed all
print(f"  Retry run: processed {len(batch2)} rows (hwm already at {state_table['last_processed_id']})")

print("\nIdempotency pattern complete.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Retry & SLA Alerting

---

```
PROBLEM:
  A task calls an external API that occasionally times out.
  Design a retry policy. Also: if the whole pipeline hasn't finished
  by 6:00 AM, alert the on-call engineer.

APPROACH:
  Retries: exponential backoff for transient failures, fixed delay for rate limits.
  SLAs: calculated from execution_date + schedule_interval + sla_offset.

RETRY CONFIG:
  retries=3            → try up to 3 extra times after failure
  retry_delay=timedelta(minutes=5)   → wait 5 min between retries
  retry_exponential_backoff=True     → double delay each retry

RETRY SCHEDULE:
  Attempt 1: fails at T+0
  Attempt 2: retry at T+5min
  Attempt 3: retry at T+10min  (with exponential: T+10min)
  Attempt 4: retry at T+20min  (with exponential: T+20min)
  FAILED permanently after attempt 4

SLA MISS DETECTION:
  SLA = execution_date + schedule_interval + sla_offset
  Example: execution_date=2024-03-15, schedule=@daily, sla=6h
  → SLA deadline = 2024-03-16 06:00:00
  If task not SUCCESS by that time → SLA miss email/callback fires

ON_FAILURE_CALLBACK:
  def on_failure(context):
      task_id = context['task_instance'].task_id
      dag_id  = context['dag'].dag_id
      send_slack_alert(f'{dag_id}/{task_id} FAILED')

KEY INSIGHT:
  Retries handle transient failures. SLAs handle stuck/slow tasks.
  Both are necessary — retries alone don't catch infinite loops or stuck processes.

TIME / SPACE:
  Exponential backoff: T_total = delay × (2^retries - 1)
  Fixed backoff: T_total = delay × retries
  Space: O(retries) — each attempt logged in DB
```


In [ ]:
# Pattern 3: Retry and SLA simulation

# Slow motion: task with transient failures and exponential backoff
# attempt 1: fails (timeout)
# attempt 2: retry after delay × 1 → fails again
# attempt 3: retry after delay × 2 → succeeds

import datetime

def run_with_retries(task_fn, retries=3, base_delay_s=5, exponential=True,
                     fail_until_attempt=2):
    """
    Simulates Airflow task retry logic with configurable backoff.
    Args:
        task_fn:           callable that may raise Exception
        retries:           max additional attempts after first failure
        base_delay_s:      base delay between retries in seconds
        exponential:       if True, doubles delay each retry
        fail_until_attempt: task succeeds on this attempt number
    """
    attempt = 0
    delay = base_delay_s
    while attempt <= retries:
        attempt += 1
        try:
            result = task_fn(attempt, fail_until_attempt)
            print(f"  Attempt {attempt}: SUCCESS → {result}")
            return result
        except Exception as e:
            if attempt > retries:
                print(f"  Attempt {attempt}: FAILED permanently → {e}")
                raise
            print(f"  Attempt {attempt}: FAILED ({e}) — retry in {delay}s")
            if exponential:
                delay *= 2  # double the wait
    return None

def api_call_task(attempt, succeed_on):
    if attempt < succeed_on:
        raise ConnectionError(f"API timeout on attempt {attempt}")
    return f"api_result_attempt_{attempt}"

print("=== Retry with Exponential Backoff ===")
print("Task: API call that fails twice, succeeds on 3rd attempt")
try:
    run_with_retries(api_call_task, retries=3, base_delay_s=5, exponential=True, fail_until_attempt=3)
except Exception:
    pass

print()
print("Task: API call that never succeeds (all retries exhausted)")
try:
    run_with_retries(api_call_task, retries=2, base_delay_s=5, exponential=True, fail_until_attempt=99)
except Exception as e:
    print(f"  Final outcome: Task permanently failed")

print()
print("=== SLA Monitoring Simulation ===")

class SLAMonitor:
    def __init__(self, sla_minutes, on_miss_callback):
        self.sla_minutes = sla_minutes
        self.on_miss_callback = on_miss_callback
        self.tasks_completed: Dict[str, datetime.datetime] = {}

    def task_completed(self, task_id, execution_date):
        self.tasks_completed[task_id] = datetime.datetime.now()

    def check_sla(self, task_id, started_at, now=None):
        now = now or datetime.datetime.now()
        deadline = started_at + datetime.timedelta(minutes=self.sla_minutes)
        if task_id not in self.tasks_completed:
            if now > deadline:
                self.on_miss_callback(task_id, deadline)
                return False
        return True

def slack_alert(task_id, deadline):
    print(f"  ⚠️  SLA MISS: {task_id} exceeded {deadline.strftime('%H:%M')} deadline")
    print(f"      Action: page on-call, investigate slow task")

monitor = SLAMonitor(sla_minutes=30, on_miss_callback=slack_alert)

# simulate task that started 35 minutes ago and hasn't finished
started = datetime.datetime.now() - datetime.timedelta(minutes=35)
fake_now = datetime.datetime.now()

print("Task 'extract' started 35 min ago (SLA=30 min):")
monitor.check_sla('extract', started, fake_now)

print()
# simulate task that finished in time
print("Task 'transform' completed 20 min after start (SLA=30 min):")
monitor.task_completed('transform', '2024-03-15')
ok = monitor.check_sla('transform', started, fake_now)
print(f"  SLA met: {ok}")

print("\nRetry and SLA pattern complete.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Dynamic DAGs — Factory Pattern

---

```
PROBLEM:
  You have 20 database tables that all need the same ETL pattern:
  extract → validate → load. Hard-coding 20 DAGs is unmaintainable.
  How do you generate them dynamically?

APPROACH:
  DAG Factory: loop over a config list and generate DAGs programmatically.
  Airflow scans the dags/ folder and picks up any DAG object at module level.

TWO APPROACHES:
  1. One DAG per table (best isolation — independent failure + schedule)
     for table in TABLE_CONFIG:
         dag = build_dag(table)  # returns DAG object
         globals()[dag.dag_id] = dag  # register at module level

  2. One DAG with TaskGroup per table (best overview in UI)
     with DAG('all_tables') as dag:
         for table in TABLE_CONFIG:
             with TaskGroup(table.name) as tg:
                 extract = build_extract_task(table)
                 validate = build_validate_task(table)
                 load = build_load_task(table)
                 extract >> validate >> load

CONFIG-DRIVEN DYNAMIC DAGs:
  TABLE_CONFIG = yaml.safe_load(open('tables.yaml'))
  or Variable.get('table_config', deserialize_json=True)

SLOW MOTION: factory for 3 tables
  config = [{'name':'orders','schedule':'@daily'},
            {'name':'users', 'schedule':'@hourly'},
            {'name':'events','schedule':'@daily'}]
  Loop → generate DAGs: etl_orders, etl_users, etl_events
  Each has identical task structure, different connection/table params.

KEY INSIGHT:
  Never duplicate DAG code — if the pattern changes, fix it in one place.
  Config-driven = table team adds a row to YAML, ETL appears automatically.

TIME / SPACE:
  DAG parsing: O(N) — N tables → N DAG objects
  Memory: each DAG object is small (~few KB) — 1000 dynamic DAGs is fine
```


In [ ]:
# Pattern 4: Dynamic DAG factory

# Slow motion: for each table config, build a DAG with identical structure
# step 1: read config (YAML/DB/Variable)
# step 2: for each table, instantiate DAG with table-specific params
# step 3: register DAG (Airflow: globals()[dag_id] = dag)
# step 4: Airflow scheduler discovers all generated DAGs

@dataclass
class TableConfig:
    name:             str
    source_schema:    str
    target_schema:    str
    schedule:         str
    partition_col:    Optional[str]
    primary_key:      str
    sla_minutes:      int = 120

class DAGFactory:
    """
    Orchestration Pattern 4 — Dynamic DAG generation.
    Approach: Config-driven factory generates N DAGs with identical task structure.
    Time:  O(N) DAGs parsed at scheduler startup
    Space: O(N × T) where T = tasks per DAG (small — few KB each)
    """
    def __init__(self, base_dag_id='etl'):
        self.base_dag_id = base_dag_id
        self.generated: List[SimpleDAG] = []

    def build_dag(self, config: TableConfig) -> SimpleDAG:
        dag_id = f"{self.base_dag_id}_{config.name}"
        dag = SimpleDAG(dag_id, config.schedule)

        # identical task structure — parameters differ per table
        tasks = [
            TaskDef(f'extract_{config.name}',
                    'python', retries=3, sla_minutes=config.sla_minutes // 3),
            TaskDef(f'validate_{config.name}',
                    'python', retries=1,
                    depends_on=[f'extract_{config.name}']),
            TaskDef(f'load_{config.name}',
                    'sql', retries=3,
                    depends_on=[f'validate_{config.name}'],
                    sla_minutes=config.sla_minutes),
            TaskDef(f'notify_{config.name}',
                    'python', retries=1,
                    depends_on=[f'load_{config.name}']),
        ]
        for t in tasks:
            dag.add_task(t)
        return dag

    def build_all(self, configs: List[TableConfig]) -> List[SimpleDAG]:
        self.generated = [self.build_dag(cfg) for cfg in configs]
        return self.generated

# simulate YAML config
TABLE_CONFIGS = [
    TableConfig('orders',     'ops',     'dwh', '@daily',  'order_date', 'order_id',  120),
    TableConfig('customers',  'ops',     'dwh', '@daily',  None,         'customer_id', 60),
    TableConfig('events',     'clickstream', 'dwh', '@hourly', 'event_date', 'event_id', 90),
    TableConfig('inventory',  'wms',     'dwh', '@daily',  'updated_at', 'sku_id',    120),
    TableConfig('shipments',  'logistics','dwh', '@daily',  'ship_date',  'shipment_id',180),
]

factory = DAGFactory()
generated_dags = factory.build_all(TABLE_CONFIGS)

print(f"=== DAG Factory: generated {len(generated_dags)} DAGs ===")
for dag in generated_dags:
    cfg = TABLE_CONFIGS[generated_dags.index(dag)]
    print(f"  {dag.dag_id:30s}: schedule={cfg.schedule:8s}  tasks={len(dag.tasks)}  sla={cfg.sla_minutes}min")

print()
print("=== Execution order for first DAG ===")
print(f"  DAG: {generated_dags[0].dag_id}")
print(f"  Order: {generated_dags[0].topological_order()}")

print()
print("=== Benefits of Factory Pattern ===")
print(f"  Lines of code (manual × {len(TABLE_CONFIGS)} tables): ~{len(TABLE_CONFIGS) * 50} lines")
print(f"  Lines of code (factory):                    ~60 lines")
print(f"  To add new table: 1 config entry → DAG auto-generated")
print(f"  To change task logic: 1 place in factory → all {len(TABLE_CONFIGS)} DAGs updated")

print("\nDynamic DAG factory pattern complete.")

<a id='9'></a>
## 9. 🧩 Pattern 5: Sensor Patterns & External Triggers

---

```
PROBLEM:
  Your ETL runs @daily but depends on an upstream vendor file that arrives
  between 01:00 and 04:00 AM unpredictably. How do you block the ETL
  until the file arrives without wasting worker slots?

APPROACH:
  Sensors poll an external condition and block the task slot until met.
  Key config: poke_interval (how often to check) + timeout (max wait).

SENSOR TYPES:
  S3KeySensor:         wait for s3://bucket/key to exist
  FileSensor:          wait for local/network file
  ExternalTaskSensor:  wait for another DAG's task to succeed
  SqlSensor:           wait for a SQL query to return rows
  HttpSensor:          wait for HTTP endpoint to return 200
  TimeSensor:          wait until a specific wall-clock time

SENSOR MODES:
  poke (default): worker holds slot while polling → wastes slot if long wait
  reschedule:     worker releases slot between polls → better resource use
  smart sensor:   centralized singleton poller (batches multiple sensors)

SLOW MOTION: S3KeySensor with reschedule mode
  poke_interval = 60s  timeout = 7200s (2 hours)
  T+0:    check s3://bucket/input.parquet → not found → release slot
  T+60s:  re-check → not found → release slot
  ...
  T+3600s: file arrives
  T+3660s: check → found! → sensor SUCCESS → downstream tasks unblock

EXTERNAL TASK SENSOR:
  waits for dag_id=upstream_dag, task_id=load, execution_date matches
  Used to chain DAGs without direct code coupling

KEY INSIGHT:
  Use mode='reschedule' for sensors with long wait times (> 5 min poke).
  It frees the worker slot between polls — avoids starving short tasks.

TIME / SPACE:
  Sensor cost: floor(wait_time / poke_interval) poll attempts
  Resource: mode='poke' = 1 worker slot held; mode='reschedule' = 0 while waiting
```


In [ ]:
# Pattern 5: Sensor simulation

# Slow motion: S3KeySensor with reschedule mode
# step 1: sensor fires → check if file exists → not found
# step 2: release worker slot → wait poke_interval
# step 3: re-schedule → check again → repeat until found or timeout
# step 4: file found → sensor succeeds → downstream tasks unblock

class SimulatedSensor:
    """
    Orchestration Pattern 5 — Sensor simulation.
    Approach: Poll external condition at intervals; release worker slot between polls.
    Time:  O(wait_time / poke_interval) poll attempts
    Space: O(1) — no state accumulated between polls
    """
    def __init__(self, sensor_id, poke_interval_s, timeout_s, mode='reschedule'):
        self.sensor_id    = sensor_id
        self.poke_interval = poke_interval_s
        self.timeout      = timeout_s
        self.mode         = mode
        self.poke_count   = 0
        self.slot_held_s  = 0  # worker slot seconds consumed

    def run(self, condition_fn, file_available_at_attempt):
        start = 0
        while start < self.timeout:
            self.poke_count += 1
            result = condition_fn(self.poke_count, file_available_at_attempt)
            if self.mode == 'poke':
                self.slot_held_s += self.poke_interval  # holds slot while waiting
            else:  # reschedule
                self.slot_held_s += 0.1  # only uses slot during the poke itself
            status = 'FOUND' if result else 'not found'
            print(f"  [{self.sensor_id}] attempt {self.poke_count:2d} (T+{start:5d}s): {status}")
            if result:
                print(f"  → SENSOR SUCCESS after {self.poke_count} attempts, {start}s elapsed")
                return True
            start += self.poke_interval
        print(f"  → SENSOR TIMEOUT after {self.timeout}s — task FAILED")
        return False

def s3_condition(attempt, available_at):
    return attempt >= available_at  # file arrives at this attempt

print("=== S3KeySensor (mode=reschedule, file arrives at attempt 5) ===")
sensor = SimulatedSensor('s3_input_sensor', poke_interval_s=60, timeout_s=7200, mode='reschedule')
sensor.run(s3_condition, file_available_at_attempt=5)
print(f"  Worker slot consumed: {sensor.slot_held_s:.1f}s (reschedule = nearly zero idle slot usage)")

print()
print("=== Same sensor with mode=poke ===")
sensor_poke = SimulatedSensor('s3_poke_sensor', poke_interval_s=60, timeout_s=7200, mode='poke')
sensor_poke.run(s3_condition, file_available_at_attempt=5)
print(f"  Worker slot consumed: {sensor_poke.slot_held_s:.0f}s ({sensor_poke.slot_held_s/60:.0f} min held — blocks other tasks!)")

print()
print("=== ExternalTaskSensor pattern ===")
print("  Wait for: dag_id='upstream_etl', task_id='load', execution_date matches")
print("  Use case: Chain DAGs without coupling code")
print("  Config:   allowed_states=['success'], execution_delta=timedelta(0)")
print("  Gotcha:   execution_date must match exactly — use execution_delta if schedules differ")

print()
print("=== SqlSensor pattern ===")
print("  Wait for: SELECT COUNT(*) FROM daily_summary WHERE date='{{ds}}' > 0")
print("  Use case: Downstream pipeline waits for upstream table to be populated")
print("  Safer than ExternalTaskSensor: checks actual data, not task status")

print()
print("=== Sensor Decision Guide ===")
sensors = [
    ('S3KeySensor',         'wait for file to appear in S3'),
    ('FileSensor',          'wait for local/NFS file'),
    ('ExternalTaskSensor',  'wait for another Airflow DAG task'),
    ('SqlSensor',           'wait for SQL query to return rows (data check)'),
    ('HttpSensor',          'wait for API endpoint to return 200'),
]
for name, use_case in sensors:
    print(f"  {name:25s}: {use_case}")

print("\nSensor pattern complete.")

<a id='10'></a>
## 10. The Orchestration Decision Map

---

```
PROBLEM                                    SOLUTION
────────────────────────────────────────────────────────────────────────────
Tasks can run simultaneously               parallel fan-out structure
Task needs upstream output                 XCom push/pull (small metadata only)
Same pattern for N tables                  DAG factory (dynamic DAGs)
Wait for external file                     S3KeySensor (mode=reschedule)
Wait for other DAG                         ExternalTaskSensor
Retry crashes on transient failure         retries=3, retry_delay, exponential=True
Alert if slow                              sla= + sla_miss_callback
Run even if upstream fails                 trigger_rule='all_done'
Re-run safe on failure                     idempotent write (overwrite partition)
Secrets in code                            Airflow Variable or AWS Secrets Manager
Different schedule per table               DAG factory with per-config schedule
Manual trigger with params                 DagRun.conf + {{ dag_run.conf }}
────────────────────────────────────────────────────────────────────────────

ORCHESTRATION TOOL COMPARISON:
  Airflow:    Python DAGs, large ecosystem, complex but flexible
  Prefect:    Python-native, easier local testing, hybrid cloud
  Dagster:    asset-based orchestration, integrated lineage, typed
  dbt:        SQL-only DAGs (models → tests → snapshots), no Python
  AWS Step Functions: serverless, JSON state machines, AWS-native
  Temporal:   code-based workflow, durable execution, microservices

CHOOSE:
  Airflow:   DE-standard, Kubernetes/Celery executors, rich operators
  Dagster:   asset-first teams, lineage visibility, modern Python
  dbt:       analytics engineers owning SQL transformations
  Step Functions: AWS-native, event-driven, low-maintenance
```


<a id='11'></a>
## 11. Interview Cheat Sheet

---

### When to reach for each pattern:

| Signal | Pattern |
|--------|----------|
| "Design an ETL pipeline" | DAG with fan-out parallel tasks |
| "Safe on failure/retry" | Idempotent writes (overwrite/upsert) |
| "Alert if slow" | SLA config + on_failure_callback |
| "Same pipeline for 20 tables" | DAG factory |
| "Wait for file/event" | Sensor (mode=reschedule) |
| "Pass data between tasks" | XCom (metadata only) |

---

### Key concepts — memorize these:

```
execution_date:   the DATA date being processed (not current time)
catchup=True:     backfill from start_date — dangerous if not idempotent
trigger_rule:     controls when task runs based on upstream state
XCom:             key-value store in Airflow DB — small objects only
Sensor reschedule: releases worker slot between polls — use for long waits
Idempotent write:  partition overwrite OR upsert on natural key
```

---

### Common templates:

```python
# TEMPLATE: Airflow DAG with retry + SLA
from datetime import timedelta
default_args = {
    'retries': 3,
    'retry_delay': timedelta(minutes=5),
    'retry_exponential_backoff': True,
    'on_failure_callback': alert_slack,
}
with DAG('etl', default_args=default_args,
         schedule_interval='@daily', catchup=False) as dag:
    ...

# TEMPLATE: DAG factory
for config in TABLE_CONFIGS:
    dag = build_dag(config)
    globals()[dag.dag_id] = dag  # Airflow discovers at module level

# TEMPLATE: Idempotent partition overwrite
df.write.format('delta').mode('overwrite') \
    .option('replaceWhere', f"date = '{execution_date}'") \
    .save(table_path)

# TEMPLATE: S3 Sensor
sensor = S3KeySensor(
    task_id='wait_for_file',
    bucket_key='s3://bucket/input/{{ ds }}.parquet',
    poke_interval=60, timeout=7200, mode='reschedule'
)
```

---

### Gotchas to not forget:

```
❌  catchup=True on non-idempotent DAGs — backfill creates duplicates
❌  Sensor mode=poke for long waits — starves worker pool
❌  XCom for large objects (DataFrames) — stored in Airflow DB, slow
❌  Hardcoded dates in DAG code — always use {{ ds }} or execution_date
✅  trigger_rule='all_done' on notification tasks — alerts fire even on failure
✅  Idempotent writes = retry-safe ETL — biggest reliability improvement
✅  DAG factory = single source of truth for all table ETL patterns
✅  SLA miss ≠ task failure — monitor both independently
```


<a id='12'></a>
## 12. Summary Map

---

```
                     🎼 ORCHESTRATION (Airflow)
                              │
        ┌─────────────────────┼──────────────────────┐
        │                     │                      │
   DAG DESIGN           RELIABILITY            SCALABILITY
   (Pattern 1)          (Patterns 2,3)         (Patterns 4,5)
        │                     │                      │
  Tasks + Ops         Idempotency             Dynamic DAGs
  Dependencies        (overwrite/upsert)       Factory pattern
  Parallel fan-out    Retry + backoff          N tables → N DAGs
  Trigger rules       SLA alerting             Config-driven
  XCom comms          on_failure_callback      Sensors
                                               (S3/SQL/ExtTask)

RELIABILITY CHECKLIST:
  □ All tasks idempotent (overwrite or upsert, not append)
  □ retries ≥ 3 for network-dependent tasks
  □ SLA configured for critical path tasks
  □ on_failure_callback sends alert
  □ No secrets in DAG code
  □ catchup=False (unless backfill is intended)
  □ Sensor mode=reschedule for long waits
  □ Notification task has trigger_rule='all_done'
```

---
*End of Orchestration Master Guide — Sean Edition*
